In [17]:
!python ../llava/eval/run_llava_3d.py \
    --model-path ChaimZhu/LLaVA-3D-7B \
    --video-path ../demo/scannet/posed_images/scene0356_00 \
    --query "Tell me the only object that I could see from the other room and describe the object."

python: can't open file '/mnt/nasA/nanri/3dvlm-attention/LLaVA-3D/../llava/eval/run_llava_3d.py': [Errno 2] No such file or directory


In [1]:
import argparse
import torch

from llava.constants import (
    IMAGE_TOKEN_INDEX,
    DEFAULT_IMAGE_TOKEN,
    DEFAULT_IM_START_TOKEN,
    DEFAULT_IM_END_TOKEN,
    IMAGE_PLACEHOLDER,
    LOC_TOKEN_INDEX,
    DEFAULT_BOX_TOKEN
)
from llava.conversation import conv_templates, SeparatorStyle
from llava.model.builder import load_pretrained_model
from llava.utils import disable_torch_init
from llava.mm_utils import (
    process_images,
    process_videos,
    tokenizer_special_token,
    get_model_name_from_path,
)

from PIL import Image

import requests
from PIL import Image
from io import BytesIO
import re

In [2]:
# Model
disable_torch_init()
torch_dtype = torch.bfloat16
mode = 'video' # mode = 'image'
model_path = 'ChaimZhu/LLaVA-3D-7B'
model_base = None
temperature = 0
top_p = None
num_beams = 1
max_new_tokens = 512


model_name = get_model_name_from_path(model_path)
tokenizer, model, processor, context_len = load_pretrained_model(
    model_path, model_base, model_name, torch_dtype=torch_dtype
)

/home/nanri/anaconda3/envs/llava-3d/lib/python3.10/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some weights of the model checkpoint at ChaimZhu/LLaVA-3D-7B were not used when initializing LlavaLlamaForCausalLM: ['model.vision_tower.vision_tower.vision_model.embeddings.class_embedding', 'model.vision_tower.vision_tower.vision_model.embeddings.patch_embedding.weight', 'model.vision_tower.vision_tower.vision_model.embeddings.position_embedding.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.layer_norm1.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.layer_norm1.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.layer_norm2.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.layer_norm2.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.mlp.fc1.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.mlp.fc1.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.mlp.fc2.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.mlp.fc2.we

In [3]:
# prompt_text = "Tell me the only object that I could see from the other room and describe the object."
# prompt_text = "The related object is located at [-0.085,1.598,1.310]. Please output the 3D bounding box of the object and then describe the object."
prompt_text = "The related object is located at [-0.83, 1.55, 1.15]. Please output the 3D bounding box of the object and then describe the object."
# prompt_text = "Please output two 3D bounding boxes of areas where are suitable for placing a cup."
# prompt_text = "Where are the possible places to put the teddy bear in the given image? Please describe all possible location."

# prompt_text = "Where is the suitable place to put the teddy bear? Please output the 3D bounding box of its location."

# prompt_text = "Where are the possible places to put the teddy bear? Please output the 3D bouding boxes of all the possible locations."
# prompt_text = "Where can a teddy bear be placed to the left of the gray box?"

# prompt_text = "Please output the 3D bounding box of the blue box."
scene_id = "scene0356_00"
video_path = f"./demo/scannet/{scene_id}"
# video_path = "./data/3rscan/4acaebc0-6c10-2a2a-852e-0226d6539299"
# embodiedscan_path = './playground/data/annotations/embodiedscan_infos_single.json'
embodiedscan_path = './playground/data/annotations/embodiedscan_infos_full.json'


qs = prompt_text
matches = re.search(r"\[([^\]]+)\]", qs)
if matches:
    coord_list = [float(x) for x in matches.group(1).split(',')]
    coord_list = [round(coord, 3) for coord in coord_list[:3]]
    qs = re.sub(r"\[([^\]]+)\]", "<boxes>", qs)
    clicks = torch.tensor([coord_list])
else:
    clicks = torch.zeros((0,3))

image_token_se = DEFAULT_IM_START_TOKEN + DEFAULT_IMAGE_TOKEN + DEFAULT_IM_END_TOKEN
if model.config.mm_use_im_start_end:
    qs = image_token_se + "\n" + qs
else:
    qs = DEFAULT_IMAGE_TOKEN + "\n" + qs

if "llama-2" in model_name.lower():
    conv_mode = "llava_llama_2"
elif "mistral" in model_name.lower():
    conv_mode = "mistral_instruct"
elif "v1.6-34b" in model_name.lower():
    conv_mode = "chatml_direct"
elif "v1" in model_name.lower():
    conv_mode = "llava_v1"
elif "3D" in model_name.lower():
    conv_mode = "llava_v1"
elif "mpt" in model_name.lower():
    conv_mode = "mpt"
else:
    conv_mode = "llava_v0"


conv = conv_templates[conv_mode].copy()
conv.append_message(conv.roles[0], qs)
conv.append_message(conv.roles[1], None)
prompt = conv.get_prompt()

if mode == 'image':
    image_files = image_parser(args)
    images = load_images(image_files)
    image_sizes = [x.size for x in images]
    images_tensor = process_images(
        images,
        processor['image'],
        model.config
    ).to(model.device, dtype=torch_dtype)
    depths_tensor = None
    poses_tensor = None
    intrinsics_tensor = None
    clicks_tensor = None

if mode == 'video':
    processor['video'].load_embodiedscan(embodiedscan_path)
    videos_dict, original_images = process_videos(
        video_path,
        processor['video'],
        mode='random',
        device=model.device,
        text=prompt_text
    )
    images_tensor = videos_dict['images'].to(model.device, dtype=torch_dtype)
    depths_tensor = videos_dict['depths'].to(model.device, dtype=torch_dtype)
    poses_tensor = videos_dict['poses'].to(model.device, dtype=torch_dtype)
    intrinsics_tensor = videos_dict['intrinsics'].to(model.device, dtype=torch_dtype)
    clicks_tensor = clicks.to(model.device, dtype=torch.bfloat16)

video_path demo/scannet/scene0356_00
video_name scannet/scene0356_00
video_folder demo
sample_frames len: 20
sample_frames: scannet/posed_images/scene0356_00/00000.jpg


In [4]:
import open3d as o3d
import numpy as np
def create_point_cloud_from_tensors(images_tensor, depths_tensor, poses_tensor, intrinsics_tensor):
    images_tensor = images_tensor[0]       # (20, 3, H, W)
    depths_tensor = depths_tensor[0]       # (20, H, W)
    poses_tensor = poses_tensor[0]         # (20, 4, 4)
    intrinsics_tensor = intrinsics_tensor[0]  # (20, 4, 4)

    H, W = depths_tensor.shape[1], depths_tensor.shape[2]
    point_clouds = []

    for i in range(images_tensor.shape[0]):
        rgb = images_tensor[i].permute(1, 2, 0).cpu().numpy()  # (H, W, 3)
        depth = depths_tensor[i].cpu().numpy()                 # (H, W)
        pose = poses_tensor[i].cpu().numpy()                   # (4, 4)
        intrinsic = intrinsics_tensor[i].cpu().numpy()[:3, :3] # (3, 3)

        # Open3D形式に変換
        rgb_np = np.ascontiguousarray((rgb * 255).astype(np.uint8))
        depth_np = np.ascontiguousarray((depth * 1000).astype(np.uint16))

        rgb_o3d = o3d.geometry.Image(rgb_np)
        depth_o3d = o3d.geometry.Image(depth_np)

        rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
            rgb_o3d, depth_o3d, depth_scale=1000.0, depth_trunc=3.0, convert_rgb_to_intensity=False)

        intrinsic_o3d = o3d.camera.PinholeCameraIntrinsic()
        intrinsic_o3d.set_intrinsics(W, H,
                                     intrinsic[0, 0], intrinsic[1, 1],
                                     intrinsic[0, 2], intrinsic[1, 2])

        pcd = o3d.geometry.PointCloud.create_from_rgbd_image(
            rgbd, intrinsic_o3d)

        # カメラ座標系 → ワールド座標系
        pcd.transform(pose)
        point_clouds.append(pcd)

    # 全点群をマージ
    full_pcd = point_clouds[0]
    for p in point_clouds[1:]:
        full_pcd += p

    return full_pcd
full_pcd = create_point_cloud_from_tensors(original_images, videos_dict['depths'], videos_dict['poses'], videos_dict['intrinsics'])

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [5]:
o3d.io.write_point_cloud(f"analysis/results/{scene_id}.ply", full_pcd)

True

In [ ]:
# create a sphere
sphere = o3d.geometry.TriangleMesh.create_sphere(radius=0.01)
sphere.translate(clicks_tensor[0].cpu().to(torch.float32).numpy())  # move to the center

# set color to green (RGB value: [0, 1, 0])
sphere.paint_uniform_color([0, 1, 0])

# save as .ply file
o3d.io.write_triangle_mesh(f"analysis/results/clicks_{scene_id}.ply", sphere)

True

In [6]:
input_ids = (
    tokenizer_special_token(prompt, tokenizer, return_tensors="pt")
    .unsqueeze(0)
    .cuda()
)

with torch.inference_mode():
    outputs = model.generate(
        input_ids,
        images=images_tensor,
        depths=depths_tensor,
        poses=poses_tensor,
        intrinsics=intrinsics_tensor,
        clicks=clicks_tensor,
        image_sizes=None,
        do_sample=True if temperature > 0 else False,
        temperature=temperature,
        top_p=top_p,
        num_beams=num_beams,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        return_dict_in_generate=True,
        output_attentions=True,
    )

output = tokenizer.batch_decode(outputs['sequences'], skip_special_tokens=True)[0].strip()
print(output)

/home/nanri/anaconda3/envs/llava-3d/lib/python3.10/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3526.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/home/nanri/anaconda3/envs/llava-3d/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:392: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/nanri/anaconda3/envs/llava-3d/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:397: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `None` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


pooled_video_features.shape torch.Size([928, 1024])
The 3D bounding box of the object is [-0.914, 1.333, 1.038, 0.381, 0.326, 0.332]. The recycling bin is blue. It is the second bin from the left.


In [ ]:
import numpy as np
import open3d as o3d
bbox_match = re.search(r'\[([\d\-\., ]+)\]', output)
if bbox_match:
    # convert bbox_match to numpy array
    bbox_str = bbox_match.group(1)
    bbox_array = np.fromstring(bbox_str, sep=',')
    print(bbox_array)  
    center_point = bbox_array[:3]
    min_bound = center_point - bbox_array[3:] / 2
    max_bound = center_point + bbox_array[3:] / 2
    print(min_bound, max_bound)

    bbox = o3d.geometry.AxisAlignedBoundingBox(min_bound, max_bound)
    bbox_lines = o3d.geometry.LineSet.create_from_axis_aligned_bounding_box(bbox)
    # o3d.visualization.draw_plotly([bbox_lines])

[-0.914  1.333  1.038  0.381  0.326  0.332]
[-1.1045  1.17    0.872 ] [-0.7235  1.496   1.204 ]


In [8]:
o3d.io.write_line_set(f"analysis/results/bbox_lines_{scene_id}.ply", bbox_lines)

True

In [9]:
from analysis.utils import (
    load_image, 
    aggregate_llm_attention, aggregate_vit_attention,
    heterogenous_stack,
    show_mask_on_image
)

In [10]:
# constructing the llm attention matrix
aggregated_prompt_attention = []
for i, layer in enumerate(outputs["attentions"][0]):
    layer_attns = layer.squeeze(0)
    attns_per_head = layer_attns.mean(dim=0)
    cur = attns_per_head[:-1].cpu().clone()
    # following the practice in `aggregate_llm_attention`
    # we are zeroing out the attention to the first <bos> token
    # for the first row `cur[0]` (corresponding to the next token after <bos>), however,
    # we don't do this because <bos> is the only token that it can attend to
    cur[1:, 0] = 0.
    cur[1:] = cur[1:] / cur[1:].sum(-1, keepdim=True)
    aggregated_prompt_attention.append(cur)
aggregated_prompt_attention = torch.stack(aggregated_prompt_attention).mean(dim=0)

# llm_attn_matrix will be of torch.Size([N, N])
# where N is the total number of input (both image and text ones) + output tokens
llm_attn_matrix = heterogenous_stack(
    [torch.tensor([1])]
    + list(aggregated_prompt_attention) 
    + list(map(aggregate_llm_attention, outputs["attentions"]))
)  # torch.Size([608, 608]) (prompt token + generated tokens, prompt token + generated tokens)

In [11]:
# identify length or index of tokens
input_token_len = model.get_video_tower().pooled_features.shape[0] + len(input_ids[0]) - 1 # -1 for the <image> token
vision_token_start = len(tokenizer(prompt.split("<image>")[0], return_tensors='pt')["input_ids"][0])
vision_token_end = vision_token_start + model.get_video_tower().pooled_features.shape[0]
output_token_len = len(outputs["sequences"][0][1:])
output_token_start = input_token_len
output_token_end = input_token_len + output_token_len

In [12]:
import numpy as np
import open3d as o3d
from collections import defaultdict
import matplotlib.pyplot as plt


def attention_heatmap_pcd(xyz, p2v, attention_weights):
    points = xyz.reshape(-1, 3)
    point_attention = attention_weights.cpu().numpy()
    point_attention = (point_attention - point_attention.min()) / (point_attention.max() - point_attention.min())


    voxel_to_points = defaultdict(list)
    for i in range(p2v.shape[0]):
        voxel_id = p2v[i].item()
        voxel_to_points[voxel_id].append(i)
    cmap = plt.cm.RdBu_r
    colors = cmap(point_attention)[:, :3]
    point_colors = np.zeros(points.shape)
    for voxel_idx, point_indices in voxel_to_points.items():
        # if len(point_indices) == 0:
        #     continue
        point_colors[point_indices] = colors[voxel_idx]

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    pcd.colors = o3d.utility.Vector3dVector(point_colors)
    return pcd



In [13]:
import matplotlib.pyplot as plt

# whether visualize the attention heatmap or 
# the image with the attention heatmap overlayed
vis_overlayed_with_attn = True

feat_xyz = model.get_video_tower().feat_xyz[0].float().cpu().numpy()
p2v = model.get_video_tower().p2v[0].cpu().numpy()

heatmap_pcds = []
output_token_inds = list(range(output_token_start, output_token_end))
for i in range(output_token_len):
    target_token_ind = output_token_inds[i]
    attn_weights_over_vis_tokens = llm_attn_matrix[target_token_ind][vision_token_start:vision_token_end]  # target generated tokenに対するvision tokenのattention行列
    # attn_weights_over_vis_tokens = attn_weights_over_vis_tokens / attn_weights_over_vis_tokens.sum()  # normalize
    heatmap_pcd = attention_heatmap_pcd(feat_xyz, p2v, attn_weights_over_vis_tokens)
    heatmap_pcds.append(heatmap_pcd)
    

In [15]:
for i in range(len(heatmap_pcds)):
    generated_token = tokenizer.decode(outputs['sequences'][0][i + 1], skip_special_tokens=True)
    print("generated token:", generated_token)
    if '/' in generated_token:
        generated_token = generated_token.replace('/', '_')
    # o3d.visualization.draw_plotly([heatmap_pcds[i], bbox_lines])
    o3d.io.write_point_cloud(f"analysis/results/heatmap_pcd_{scene_id}_{i}_{generated_token}.ply", heatmap_pcds[i])

generated token: The
generated token: 
generated token: 3
generated token: D
generated token: bound
generated token: ing
generated token: box
generated token: of
generated token: the
generated token: object
generated token: is
generated token: [-
generated token: 0
generated token: .
generated token: 9
generated token: 3
generated token: 4
generated token: ,
generated token: 
generated token: 1
generated token: .
generated token: 3
generated token: 3
generated token: 3
generated token: ,
generated token: 
generated token: 1
generated token: .
generated token: 0
generated token: 1
generated token: 8
generated token: ,
generated token: 
generated token: 0
generated token: .
generated token: 3
generated token: 8


generated token: 1
generated token: ,
generated token: 
generated token: 0
generated token: .
generated token: 3
generated token: 2
generated token: 6
generated token: ,
generated token: 
generated token: 0
generated token: .
generated token: 3
generated token: 3
generated token: 8
generated token: ].
generated token: The
generated token: rec
generated token: y
generated token: cling
generated token: bin
generated token: is
generated token: blue
generated token: .
generated token: It
generated token: is
generated token: the
generated token: second
generated token: bin
generated token: from
generated token: the
generated token: left
generated token: .
generated token: </s>


In [19]:
from analysis.make_pcd import create_pointcloud_from_rgbd
import cv2
import json

color_img = cv2.imread("demo/scannet/posed_images/scene0356_00/00000.jpg")
color_img = cv2.cvtColor(color_img, cv2.COLOR_BGR2RGB)
color_img = cv2.resize(color_img, (640, 480))
depth_img = cv2.imread("demo/scannet/posed_images/scene0356_00/00000.png", cv2.IMREAD_ANYDEPTH)
depth_img = depth_img.astype(np.float32)

with open('playground/data/annotations/embodiedscan_infos_single.json', 'r') as f:
    data = json.load(f)
    
# シーンIDと画像IDからdepth_intrinsicを取得
depth_intrinsic = np.array(data['scannet/scene0356_00']['depth_intrinsic'])

# 4x4行列から3x3行列に変換（最後の行と列を削除）
depth_intrinsic = depth_intrinsic[:3, :3]
pcd = create_pointcloud_from_rgbd(color_img, depth_img, depth_intrinsic)